\*Please Run in colab

# Setup

### Environment Setup

In [1]:
# ! apt-get install -y libgraphviz-dev
# ! pip install pygraphviz

In [2]:
!pip install instructor

In [3]:
!pip install feedparser

In [4]:
!pip install langchain langchain-openai

In [5]:
!cd /content
!rm -rf macro_financial_forecasting

In [6]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 2241, done.
remote: Counting objects: 100% (726/726), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 2241 (delta 556), reused 511 (delta 493), pack-reused 1515 (from 3)
Receiving objects: 100% (2241/2241), 39.99 MiB | 13.64 MiB/s, done.
Resolving deltas: 100% (1463/1463), done.


In [7]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


### Code Setup

In [8]:
from config import Config
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")
print(f"Config: {config}")

Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'https://feeds.bloomberg.com/real-e

# Agent

#Langgraph

In [9]:
from langgraph.pipeline import build_graph, create_initial_state
import nest_asyncio
nest_asyncio.apply()

Device set to use cuda:0


In [10]:
# app = build_graph()
# MODE = "mock"  # or "real"

# result = await app.ainvoke(create_initial_state(mode=MODE))

# print(result["predictions"])  # Display the final predictions dataframe
# print(result["summary"])

Raw sentiment rows: 3
Unique mapped tickers in sentiment: ['SPY', 'XLF']
Tickers: ['SPY', 'XLF']
OHLCV date range: 2025-08-29 23:44:21 to 2025-11-27 23:44:21


[*********************100%***********************]  2 of 2 completed
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


OHLCV shape: (62, 10)
OHLCV columns (first few): [('Close', 'SPY'), ('Close', 'XLF'), ('High', 'SPY'), ('High', 'XLF'), ('Low', 'SPY'), ('Low', 'XLF'), ('Open', 'SPY'), ('Open', 'XLF')]
ohlcv_long shape: (124, 4)
model_df shape after OHLCV merge: (3, 20)
Dropped 0 rows due to missing MKT or SentimentScore.
Numeric feature matrix shape: (3, 4)
         Industry                Date  \
0  General Market 2025-11-27 22:29:53   
1      Financials 2025-11-27 23:33:36   
2      Financials 2025-11-27 23:44:21   

                                                News  ArticleCount  \
0  [{'Headline': 'Asian Stocks Ebb as Global Rall...             1   
1  [{'Headline': 'Oil in Worst Monthly Run Since ...             1   
2  [{'Headline': 'Gold Poised for Fourth Monthly ...             1   

                                       ImpactfulNews  AvgSentimentScore  \
0  [{'Headline': 'Asian Stocks Ebb as Global Rall...           0.003999   
1  [{'Headline': 'Oil in Worst Monthly Run Since ...       

In [11]:
# from IPython.display import display, Image

# # Get the graph object
# graph = app.get_graph()

# # Render the graph as a PNG image in memory and display it
# display(Image(graph.draw_png()))

# Streamlit Tunnel

In [10]:
 !pip install -q streamlit

In [11]:
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [12]:
%%writefile app.py
import streamlit as st
import nest_asyncio
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import asyncio

from langgraph.pipeline import build_graph, create_initial_state

# Enable asyncio in Streamlit/Colab
nest_asyncio.apply()

# Page config
st.set_page_config(
    page_title="📈 Financial News Pipeline",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="expanded",
)

# Title and intro
st.title("🚀 Agentic Factor Extraction from Financial News Streams")
st.markdown("**Interactive LangGraph visualization** - Track each stage and final predictions")

# Sidebar controls
st.sidebar.header("📋 Pipeline Controls")

mode = st.sidebar.selectbox(
    "Mode",
    options=["mock", "real"],
    index=0,
)

days_back = st.sidebar.slider(
    "Days back (RSS)",
    min_value=1,
    max_value=7,
    value=1,
    step=1,
)

run_clicked = st.sidebar.button("🚀 Run Pipeline", type="primary")

# Initialize session state
if "result" not in st.session_state:
    st.session_state.result = None
if "mode" not in st.session_state:
    st.session_state.mode = mode
if "days_back" not in st.session_state:
    st.session_state.days_back = days_back
if "app" not in st.session_state:
    st.session_state.app = build_graph()

# Run pipeline
if run_clicked:
    with st.spinner("Running pipeline... This may take 30–60 seconds"):
        try:

            initial_state = create_initial_state(mode=mode, days_back=days_back)
            result = asyncio.run(st.session_state.app.ainvoke(initial_state))
            st.session_state.result = result
            st.session_state.mode = mode
            st.session_state.days_back = days_back
            st.success("✅ Pipeline completed!")
        except Exception as e:
            st.error(f"❌ Pipeline failed: {e}")
            st.exception(e)

# Show results
if st.session_state.result is not None:
    result = st.session_state.result

    # Pipeline execution summary
    st.header("📊 Pipeline Execution")
    col1, col2 = st.columns([1, 3])

    with col1:
        st.subheader("Execution Flow")
        nodes = ["route", "load_*", "process_*", "predict", "summarize", "FINAL"]
        colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7", "#DDA0DD"]

        fig_flow = go.Figure(
            go.Funnel(
                y=nodes,
                x=[1] * len(nodes),
                marker={"color": colors},
                text=["Entry", "Load News", "Process", "Predict", "Summarize", "Result"],
                textinfo="text",
            )
        )
        fig_flow.update_layout(height=350, title="Pipeline Steps", showlegend=False)
        st.plotly_chart(fig_flow, use_container_width=True)

    with col2:
        st.subheader("Execution Info")
        st.metric("Mode", st.session_state.mode.upper())
        st.metric("Days Back", f"{st.session_state.days_back} days")
        st.metric("News Articles", len(result.get("raw_data", [])))
        st.metric(
            "Predictions",
            len(result.get("predictions", [])) if result.get("predictions") is not None else 0,
        )

    # Node-by-node trace (simple)
    st.header("🔍 Node-by-Node Execution")
    trace_data = []
    state_keys = ["raw_data", "processed_data", "predictions", "summary"]
    for key in state_keys:
        if result.get(key) is not None:
            node_name = key.replace("_", " ").title()
            val = result[key]
            data_size = len(val) if hasattr(val, "__len__") and not isinstance(val, str) else "N/A"
            trace_data.append(
                {
                    "Node": node_name,
                    "Data Size": data_size,
                    "Status": "✅ Complete",
                }
            )
    if trace_data:
        st.dataframe(pd.DataFrame(trace_data), use_container_width=True)

    # Tabs with detailed outputs
    tab1, tab2, tab3, tab4, tab5 = st.tabs(
        ["🎯 AI Summary", "📈 Predictions", "📰 Processed News", "📋 Raw Data", "💬 Messages"]
    )

    with tab1:
        st.header("🤖 Quantitative Analyst Report")
        if result.get("summary"):
            st.markdown(result["summary"])
        else:
            st.warning("No summary generated yet")

        if isinstance(result.get("predictions"), pd.DataFrame):
            df = result["predictions"]
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                st.metric("Avg Predicted Return", f"{df['pred_ret_next'].mean():.2%}")
            with col2:
                st.metric("Best Prediction", f"{df['pred_ret_next'].max():.2%}")
            with col3:
                st.metric("Avg Sentiment", f"{df['SentimentScore'].mean():.3f}")
            with col4:
                st.metric("Industries", df["Industry"].nunique())

    with tab2:
        st.header("📊 Next-Day Return Predictions")
        if isinstance(result.get("predictions"), pd.DataFrame) and not result["predictions"].empty:
            df = result["predictions"].copy()

            # Aggregate to one row per industry (e.g. mean predicted return)
            agg = (
                df.groupby("Industry", as_index=False)
                  .agg(pred_ret_next=("pred_ret_next", "mean"),
                      SentimentScore=("SentimentScore", "mean"))
            )
            agg["pred_ret_next_pct"] = agg["pred_ret_next"] * 100

            st.dataframe(
                agg[["Industry", "pred_ret_next_pct", "SentimentScore"]].round(2),
                use_container_width=True,
            )

            # Bar chart: ALL industries present in agg
            fig = px.bar(
                agg,
                x="Industry",
                y="pred_ret_next_pct",
                color="pred_ret_next_pct",
                color_continuous_scale=["#d73027", "#ffffbf", "#1a9850"],  # red→yellow→green
                title="Predicted Returns by Industry",
                labels={"pred_ret_next_pct": "Predicted Return (%)"},
            )
            fig.update_coloraxes(cmid=0)  # 0 = center of diverging scale
            fig.update_traces(texttemplate="%{y:.1f}%", textposition="auto")
            fig.update_layout(yaxis_tickformat=".1f")
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.warning("No predictions available")

    with tab3:
        st.header("🔬 Processed News Data")
        if isinstance(result.get("processed_data"), pd.DataFrame):
            df = result["processed_data"]

            # Show main numeric/text columns first (hide complex list columns)
            base_cols = [
                col for col in df.columns
                if col not in ["News", "ImpactfulNews"]
            ]
            st.subheader("Summary table")
            st.dataframe(df[base_cols], use_container_width=True)

            # Optional: sentiment histogram
            if "SentimentScore" in df.columns:
                fig_sent = px.histogram(
                    df,
                    x="SentimentScore",
                    color="Industry",
                    title="Sentiment Score Distribution",
                    nbins=12,
                )
                st.plotly_chart(fig_sent, use_container_width=True)
        else:
            st.warning("No processed data available")

    with tab4:
        st.header("📰 Raw Bloomberg RSS Data")
        raw_data = result.get("raw_data", [])
        if raw_data:
            st.write(f"**{len(raw_data)} articles loaded**")

            for i, article in enumerate(raw_data[:60]):  # cap to first 60 for speed
                # Handle both Pydantic model and dict
                if hasattr(article, "model_dump"):
                    a = article.model_dump()
                elif isinstance(article, dict):
                    a = article
                else:
                    a = {}

                headline = a.get("Headline") or a.get("headline") or "Untitled"
                date = a.get("Date") or a.get("date") or ""
                link = a.get("Link") or a.get("link")
                body = a.get("Article") or a.get("article") or ""

                with st.expander(f"Article {i+1}: {headline}"):
                    # Headline & meta
                    st.markdown(f"**Headline:** {headline}")
                    if date:
                        st.markdown(f"**Date:** {date}")
                    if link:
                        st.markdown(f"**Link:** {link}")
                    st.markdown(f"**Content:** {body}")

            if len(raw_data) > 60:
                st.info(f"... and {len(raw_data) - 60} more articles")
        else:
            st.warning("No raw data available")

    with tab5:
        st.header("💬 Agent Messages Log")
        messages = result.get("messages", [])
        for i, msg in enumerate(messages[-10:]):
            with st.expander(f"Message {len(messages) - len(messages[-10:]) + i + 1}"):
                st.write(msg.content if hasattr(msg, "content") else str(msg))

# Sidebar instructions
with st.sidebar.expander("How to Use"):
    st.markdown(
        """
1. Choose mode:
   - `mock` = fast demo data
   - `real` = live RSS + yfinance
2. Adjust days back for RSS feeds
3. Click Run to execute the pipeline
4. Explore tabs for detailed outputs
"""
    )

st.markdown("---")
st.markdown("*Built for quantitative finance analysis | © Chuan Bin Phoe and Neaton Ang*")


Writing app.py


In [13]:
!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501 --subdomain macro-economic-forecasting

⠙your url is: https://macro-economic-forecasting.loca.lt
^C
